# 蜃景 × lightx2v —— 纯文生视频 (t2v) Colab 部署（无 ComfyUI）

**怎么用：菜单「代码执行程序 → 全部运行」(Run all)。** 断线/回收后再 Run all——Wan 权重 + lightx2v（含 patch + config）全持久化 Drive，**不重下、不重 clone、不重 patch**，几分钟恢复。

**跑前一次性：**
1. 运行时选 **GPU**（Blackwell / Hopper / Ada / A100 都行）。
2. 右侧 🔑 Secrets 可加 **HF_TOKEN**（下 Wan 权重快些；公开模型匿名也能下）。
3. 第 1 格 `DEEPSEEK_KEY` 可留空（分镜 / AI 分析用前端配的 grok 模型）。

**本笔记本已固化的踩坑修复（不用再手动改）：**
- lightx2v 在 **Blackwell(sm_120)** 上硬依赖 flash_attn 装不上 → §3 自动 **patch worldmirror 的 flash_attn import**，绕过即可 import（我们不用 worldmirror）。
- runner 用 **`wan2.2_moe`**（A14B 双专家，不是 `wan2.2`）、attention 用 **`torch_sdpa`**（免 flash_attn）、`cpu_offload` 按显存自动。
- lightx2v 用**普通 pip**（不用 editable——Drive FUSE 上 editable 会崩）。
- 工作台前端已是**纯 t2v**：粘小说 → 一键分析 → 每镜「出片(t2v)」，没有出图 / 选图。

In [ ]:
# === §1 参数 + Drive + GPU 探测 ===
import os, sys
DEEPSEEK_KEY = ''            # 分镜/AI 分析用;留空=用前端配的 grok。也可填 OpenRouter/grok key
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'; BRANCH = 'main'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'; CACHE = DRIVE + '/mirage_models'; TOOLS = DRIVE + '/mirage_tools'
for p in (CACHE, TOOLS):
    os.makedirs(p, exist_ok=True)
os.environ['HF_HOME'] = TOOLS + '/hf_cache'; os.makedirs(os.environ['HF_HOME'], exist_ok=True)
!nvidia-smi -L
import torch
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cc = torch.cuda.get_device_capability(0)
    os.environ['LX_CPU_OFFLOAD'] = 'false' if gb > 70 else 'true'   # >70G 两专家全 GPU;否则挪 CPU
    print(f'GPU {torch.cuda.get_device_name(0)} sm{cc[0]}.{cc[1]} {gb:.0f}G -> cpu_offload={os.environ["LX_CPU_OFFLOAD"]}')
    if cc[0] >= 12:
        print('  Blackwell(sm120): lightx2v flash_attn 装不上正常,§3 已 patch 绕过、走 torch_sdpa。')
else:
    os.environ['LX_CPU_OFFLOAD'] = 'true'; print('没检测到 GPU! 代码执行程序 -> 更改运行时类型 -> 选 GPU')
# 子进程继承内核 torch(split-torch 环境必需)
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in sys.path if p)
print('环境就绪 | 模型:', CACHE, '| 工具:', TOOLS)

In [ ]:
# === §2 拉取/更新 mirage 仓库 ===
import os, sys
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')
print('mirage 就绪 @', BRANCH)

In [ ]:
# === §3 装 lightx2v（clone 持久化 Drive + patch worldmirror + 普通 pip + 改 config）★幂等可重跑 ===
# Blackwell(sm120) 上 lightx2v 的 worldmirror 模块硬 import flash_attn,装不上→把那两行 import 改成 None
# (我们只跑 Wan t2v,不用 worldmirror)。attention 走 torch_sdpa,免 flash_attn。
import os, subprocess, sys, json, importlib.util as u
sys.path.insert(0, '/content/mirage/colab'); import persist
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
LX_DRIVE = TOOLS + '/LightX2V'; LX = '/content/LightX2V'
# 1) 仓库(持久化 Drive,不重 clone)
if not os.path.isdir(LX_DRIVE + '/.git'):
    sh(f'git clone https://github.com/ModelTC/lightx2v {LX_DRIVE}')
persist.link_dir(LX_DRIVE, LX)
# 2) patch worldmirror 的 flash_attn 硬 import(幂等;在 pip 装之前先 patch clone,装进去的就是 patch 过的)
fa = LX + '/lightx2v/models/networks/worldmirror/models/layers/attention.py'
if os.path.exists(fa):
    s = open(fa).read()
    s = s.replace('from flash_attn_interface import flash_attn_func as flash_attn_func_v3', 'flash_attn_func_v3 = None  # [patch sm120]')
    s = s.replace('from flash_attn.flash_attn_interface import flash_attn_func as flash_attn_func_v2', 'flash_attn_func_v2 = None  # [patch sm120]')
    open(fa, 'w').write(s)
# 3) 普通 pip 装(从 patch 过的 clone;★不用 editable,FUSE 上会崩)
if not u.find_spec('lightx2v'):
    print('装 lightx2v(普通 pip)...'); print(sh(f'pip install {LX} 2>&1 | tail -6'))
# 4) config: torch_sdpa(免 flash_attn) + cpu_offload 按显存
cfg = LX + '/configs/wan/wan_t2v.json'; c = json.load(open(cfg))
for k in ('self_attn_1_type', 'cross_attn_1_type', 'cross_attn_2_type'):
    c[k] = 'torch_sdpa'
c['cpu_offload'] = (os.environ.get('LX_CPU_OFFLOAD', 'true') == 'true'); c['offload_granularity'] = 'block'
json.dump(c, open(cfg, 'w'), indent=4)
print('lightx2v 可导入:', bool(u.find_spec('lightx2v')), '| config:', {k: c[k] for k in ('self_attn_1_type', 'cpu_offload')})

In [ ]:
# === §4 下权重（Wan-AI base 双专家 + lightx2v 4步蒸馏 LoRA，持久化 Drive，幂等不重下）===
import os, glob
MODEL = CACHE + '/wan2.2_t2v_a14b'; LORA = CACHE + '/lightx2v_t2v_lora'
os.makedirs(MODEL, exist_ok=True); os.makedirs(LORA, exist_ok=True)
# base: Wan-AI/Wan2.2-T2V-A14B 高/低噪双专家(各~28G) + VAE + 文本编码器。缺啥补啥(幂等)。
!hf download Wan-AI/Wan2.2-T2V-A14B --include "high_noise_model/*" "low_noise_model/*" "*.json" "Wan2.1_VAE.pth" "models_t5_umt5-xxl-enc-bf16.pth" "google/*" --local-dir "{MODEL}"
# 4步蒸馏 LoRA(小;给 lightx2v 4步加速,可选)
_LORA = 'Wan2.2-T2V-A14B-4steps-lora-rank64-Seko-V2.0'
!hf download lightx2v/Wan2.2-Lightning --include "{_LORA}/*" --local-dir "{LORA}"
hi = len(glob.glob(f'{MODEL}/high_noise_model/*.safetensors')); lo = len(glob.glob(f'{MODEL}/low_noise_model/*.safetensors'))
os.environ['LIGHTX2V_MODEL_T2V'] = MODEL
print('权重 high/low:', hi, '/', lo, '✅' if (hi and lo) else '❌ 缺权重→重跑本格', '| LIGHTX2V_MODEL_T2V =', MODEL)

In [ ]:
# === §5 起 lightx2v server（model_cls=wan2.2_moe；端口 8189；脱离内核,停 cell 不杀）===
import os, subprocess, sys
sys.path.insert(0, '/content/mirage/colab'); import persist
LX = '/content/LightX2V'; MODEL = CACHE + '/wan2.2_t2v_a14b'
subprocess.run("fuser -k 8189/tcp 2>/dev/null; pkill -9 -f 'lightx2v.server' 2>/dev/null; true", shell=True)  # 硬重启
e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'
open('/content/lightx2v.log', 'w').close()
subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 't2v',
                  '--model_path', MODEL, '--config_json', f'{LX}/configs/wan/wan_t2v.json',
                  '--host', '0.0.0.0', '--port', '8189'],
                 cwd=LX, env=e, stdout=open('/content/lightx2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
print('lightx2v server 起中(首次加载 A14B 双专家较慢,最多等 5 分钟)...')
ok = persist.wait_http('http://127.0.0.1:8189/v1/service/status', timeout=300)
print('✅ lightx2v 就绪' if ok else '✗ 未就绪 → 看下面日志')
print(subprocess.run('tail -40 /content/lightx2v.log', shell=True, capture_output=True, text=True).stdout)
print('\n★若崩在 KeyError patch_embedding / 加载权重失败:权重是 diffusers 格式 → 跑 §5b 转格式,再回本格。')

In [ ]:
# === §5b (仅当 §5 崩在「KeyError patch_embedding / 加载权重」时跑) diffusers→native 转格式 ===
import glob
LX = '/content/LightX2V'; MODEL = CACHE + '/wan2.2_t2v_a14b'
for d in ('high_noise_model', 'low_noise_model'):
    !cd {LX} && python tools/convert/converter.py --source {MODEL}/{d}/ --output {MODEL}/{d}/ --output_ext .safetensors --output_name wan_{d} --model_type wan_dit --single_file
f = glob.glob(f'{MODEL}/high_noise_model/wan_*.safetensors')
if f:
    from safetensors import safe_open
    print('转后 key 样例:', list(safe_open(f[0], 'pt').keys())[:5])
print('转完 → 回 §5 重起 server')

In [ ]:
# === §6 装后端依赖 + 写 .env（纯 t2v：COMFYUI 空、lightx2v 接通）===
import os, re, shutil
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
shutil.copy('.env.colab', '.env'); env = open('.env').read()
def _se(e, k, v): return re.sub(k + r'=.*', k + '=' + v, e) if (k + '=') in e else e + ('' if e.endswith(chr(10)) else chr(10)) + k + '=' + v + chr(10)
MODEL = CACHE + '/wan2.2_t2v_a14b'
_LD = CACHE + '/lightx2v_t2v_lora/Wan2.2-T2V-A14B-4steps-lora-rank64-Seko-V2.0'
for k, v in [('COMFYUI_BASE_URL', ''), ('LIGHTX2V_ENABLED', 'true'), ('LIGHTX2V_BASE_URL', 'http://127.0.0.1:8189'),
             ('LIGHTX2V_MODEL_T2V', MODEL), ('T2V_PROVIDER', 'lightx2v-t2v'), ('VIDEO_PROVIDER_DEFAULT', 'wan2.2'),
             ('WAN_T2V_LORA_HIGH', _LD + '/high_noise_model.safetensors'),
             ('WAN_T2V_LORA_LOW', _LD + '/low_noise_model.safetensors')]:
    env = _se(env, k, v)
if DEEPSEEK_KEY:
    env = _se(env, 'OPENAI_API_KEY', DEEPSEEK_KEY)
open('.env', 'w').write(env)
print('.env 写好(纯 t2v: COMFYUI 空 / lightx2v 接通 / 蒸馏 LoRA 登记)')

In [ ]:
# === §7 构建前端（纯 t2v 工作台）===
%cd /content/mirage/frontend
!npm install --silent && npm run build || echo '前端构建失败(用 API 也行)'
%cd /content/mirage

In [ ]:
# === §8 起后端（硬重启：杀旧+起新，读 §6 的 .env）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端', ['uvicorn', 'mirage.main_api:app', '--host', '0.0.0.0', '--port', '8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)

In [ ]:
# === §9 cloudflared 暴露后端 → 公网 URL ===
import subprocess, re, time, os, sys
sys.path.insert(0, '/content/mirage/colab'); import persist
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
open('/content/cf.log', 'w').close()
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
url = None
for _ in range(60):
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m:
        url = m[-1]; break
print('✅ 公网地址:', url if url else '⚠ 60s 未就绪,重跑本格')
print('打开它 → 工作台已是纯 t2v: 粘小说 → 一键分析 → 每镜「出片(t2v)」(走 lightx2v,无 ComfyUI)')

In [ ]:
# === §10 实时日志（lightx2v 出片采样进度；停本格只停 tail、不杀 server）===
!tail -n 40 -f /content/lightx2v.log

---
## 角色 LoRA 训练（Wan-T2V，可选 · 独立）

训练用 `colab_deploy.ipynb` 的 **LW1/LW2**（ai-toolkit，与 lightx2v 推理是两套，互不影响）。
训好的 `角色_wan_t2v_high/low.safetensors` 拷进 `mirage_models/loras/`，在工作台项目里填上（或 `.env` 的 `WAN_T2V_LORA_HIGH/LOW`）即可。仅原创虚构成年角色，遵守合规前置。

## 首跑核对清单（出问题按这个排）
1. **§3** `lightx2v 可导入: True`？否 → 看 pip 报错。
2. **§4** `权重 high/low: 6 / 6`？否 → 重跑 §4（断点续下）。
3. **§5** `✅ lightx2v 就绪`？崩在 `KeyError patch_embedding` → 跑 §5b 转格式再回 §5。
4. **§8/§9** 后端起 + 公网 URL 出来 → 打开出片。